#### Generating Ground Truth Data

In [ ]:
#Loading the documents
from ingest import load_faq_data

documents = load_faq_data()

In [2]:
documents_llm = [x for x in documents if x['course'] == 'llm-zoomcamp']

len(documents), len(documents_llm)

(1375, 113)

In [3]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

9e508f2212
Course: When does the course start?
A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.


In [ ]:
# Generating questions with structured output
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {'role': 'developer', 'content': data_gen_instructions}, 
    {'role': 'user', 'content': user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

print(result)

questions=['When does the Data Engineering Zoomcamp usually kick off each year, and where can I find the exact date for the current run?', 'How do I register for the next cohort before it starts, and what link should I use?', 'Is there a set time of year for this course, or does it change every year?', 'Where should I check for the official start date and sign-up info for the current cohort?', 'What are the best channels to follow if I want announcements about the course start and registration?']


In [13]:
print(result.questions)

['When does the Data Engineering Zoomcamp usually kick off each year, and where can I find the exact date for the current run?', 'How do I register for the next cohort before it starts, and what link should I use?', 'Is there a set time of year for this course, or does it change every year?', 'Where should I check for the official start date and sign-up info for the current cohort?', 'What are the best channels to follow if I want announcements about the course start and registration?']


In [14]:
# Reusable utilities
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client, 
    data_gen_instructions, 
    user_prompt, 
    Questions
)

In [15]:
result.questions

['When does the data engineering zoomcamp usually begin each year?',
 'Where can I find the exact start date for the current cohort?',
 'How do I register for the upcoming cohort before it starts?',
 'Is there a place to get announcements about the course start date?',
 'What Telegram or Slack channels should I join for course updates?']

In [16]:
# Tracking cost
usage.input_tokens, usage.output_tokens

(297, 78)

In [17]:
from evaluation_utils import calc_price

In [18]:
cost = calc_price(usage)
cost

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.00035099999999999997,
 'total_cost': 0.00057375}

In [19]:
# Now convert these questions into ground truth records:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'When does the data engineering zoomcamp usually begin each year?',
  'document': '9e508f2212'},
 {'question': 'Where can I find the exact start date for the current cohort?',
  'document': '9e508f2212'},
 {'question': 'How do I register for the upcoming cohort before it starts?',
  'document': '9e508f2212'},
 {'question': 'Is there a place to get announcements about the course start date?',
  'document': '9e508f2212'},
 {'question': 'What Telegram or Slack channels should I join for course updates?',
  'document': '9e508f2212'}]

#### Generating Ground Truth for All Documents

In [20]:
from evaluation_utils import llm_structured_retry

In [21]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        openai_client, 
        data_gen_instructions, 
        user_prompt, 
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage


In [22]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

100%|██████████| 5/5 [00:11<00:00,  2.38s/it]


In [25]:
# Parallel processing
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [35]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents_llm, generate_ground_truth)

100%|██████████| 113/113 [00:40<00:00,  2.76it/s]


In [36]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [37]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08898450000000001

In [38]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08898450000000001

In [39]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [40]:
df_ground_truth.to_csv("../../../data/ground_truth-new.csv", index=False)